# 02 — Pipeline A: componentes (H3)

Hito **H3** del proyecto ABP.

Demuestra cada bloque del pipeline clásico sobre imágenes reales del test:

1. **Selective Search** — region proposals sin clase.
2. **HOG** — descriptor de forma global sobre un crop.
3. **SIFT** — keypoints locales sobre el mismo crop.
4. **BoVW** — histograma de palabras visuales usando el codebook entrenado.
5. **Feature final** — concatenación HOG + BoVW (vector usado por la SVM en H4).

**Requisitos previos**:

- H1 y H2 cerrados (splits en `data/processed/`).
- Codebook entrenado: `uv run python scripts/train_codebook.py` (produce `data/processed/codebook.pkl`).

Este notebook **no entrena la SVM** — eso es H4. Solo verifica que cada pieza funciona y mide tiempos.

In [1]:
import sys
import time
from pathlib import Path

notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(repo_root / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from grocery_detection.utils.config import load_yaml
from grocery_detection.data.eda import load_coco
from grocery_detection.classical.proposals import (
    resize_for_proposals,
    scale_proposals,
    selective_search,
)
from grocery_detection.classical.descriptors.hog import HOG_TARGET_SIZE, compute_hog, hog_dim
from grocery_detection.classical.descriptors.sift import compute_sift
from grocery_detection.classical.descriptors.bovw import encode_bovw, load_codebook
from grocery_detection.data.visualize import draw_bbox

data_cfg = load_yaml(repo_root / "configs" / "data.yaml")
img_dir = repo_root / data_cfg["paths"]["d2s_images"]
test_coco = load_coco(repo_root / data_cfg["filtered_splits"]["test"])
codebook_path = repo_root / "data" / "processed" / "codebook.pkl"
print(f"Test images: {len(test_coco['images'])}")
print(f"Codebook   : {codebook_path}  (existe={codebook_path.exists()})")

ModuleNotFoundError: No module named 'skimage'

## 1. Selective Search

Generamos region proposals sobre una escena cluttered del test. Mostramos solo los top-50 para no saturar la figura. SS realmente produce ~1000 candidatos en modo *fast*.

In [ ]:
# Elegir una imagen test con varios objetos
anns_by_img = {}
for a in test_coco["annotations"]:
    anns_by_img.setdefault(a["image_id"], []).append(a)
id2img = {im["id"]: im for im in test_coco["images"]}

# La primera imagen test con >= 4 objetos
chosen = next(im for im in test_coco["images"] if len(anns_by_img.get(im["id"], [])) >= 4)
print(f"Imagen elegida: {chosen['file_name']}  ({len(anns_by_img[chosen['id']])} GT objects)")

img_bgr = cv2.imread(str(img_dir / chosen["file_name"]))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
print(f"Image shape: {img_bgr.shape}")

# SS sobre versión reducida (640px lado largo). Aceleramos sin perder casi nada de calidad.
resized, scale = resize_for_proposals(img_bgr, max_side=640)
t0 = time.time()
props_small = selective_search(resized, mode="fast", max_proposals=None)
print(f"SS fast: {len(props_small)} proposals en {time.time()-t0:.2f}s (image scale={scale:.3f})")

# Reescalar proposals al tamaño original
props = scale_proposals(props_small, scale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Izquierda: GT bboxes
gt_img = img_rgb.copy()
id2name = {c["id"]: c["name"] for c in test_coco["categories"]}
for ann in anns_by_img[chosen["id"]]:
    x, y, w, h = ann["bbox"]
    draw_bbox(gt_img, x, y, w, h, label=id2name[ann["category_id"]], color=(0, 200, 0))
axes[0].imshow(gt_img)
axes[0].set_title(f"Ground truth ({len(anns_by_img[chosen['id']])} obj.)")
axes[0].axis("off")

# Derecha: top-50 proposals
prop_img = img_rgb.copy()
for (x, y, w, h) in props[:50]:
    cv2.rectangle(prop_img, (int(x), int(y)), (int(x+w), int(y+h)), (255, 0, 0), 2)
axes[1].imshow(prop_img)
axes[1].set_title(f"Top-50 SS proposals (de {len(props)} totales)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 2. HOG

Tomamos el bounding box del primer objeto GT como ejemplo y extraemos su HOG. Visualizamos la rejilla de gradientes.

In [ ]:
first_ann = anns_by_img[chosen["id"]][0]
first_label = id2name[first_ann["category_id"]]
bbox_xywh = tuple(int(v) for v in first_ann["bbox"])
print(f"Objeto elegido: {first_label}, bbox={bbox_xywh}")

hog_vec, hog_viz = compute_hog(img_bgr, bbox=bbox_xywh, visualize=True)
print(f"HOG vector: shape={hog_vec.shape}  (esperado {hog_dim()})")

x, y, w, h = bbox_xywh
crop_rgb = img_rgb[max(0,y):y+h, max(0,x):x+w]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(crop_rgb); axes[0].set_title(f"Crop original ({crop_rgb.shape[1]}x{crop_rgb.shape[0]})"); axes[0].axis("off")
axes[1].imshow(cv2.resize(crop_rgb, HOG_TARGET_SIZE)); axes[1].set_title(f"Resized {HOG_TARGET_SIZE[0]}x{HOG_TARGET_SIZE[1]}"); axes[1].axis("off")
axes[2].imshow(hog_viz, cmap="gray"); axes[2].set_title("HOG (gradientes)"); axes[2].axis("off")
plt.tight_layout()
plt.show()

## 3. SIFT

Sobre el mismo crop, SIFT detecta keypoints locales (esquinas, bordes con orientación). Cada keypoint da un vector 128-D invariante a escala/rotación.

In [ ]:
kp, desc = compute_sift(img_bgr, bbox=bbox_xywh)
print(f"SIFT: {len(kp)} keypoints, descriptors shape={desc.shape}")

crop_for_viz = img_bgr[max(0,y):y+h, max(0,x):x+w].copy()
crop_with_kp = cv2.drawKeypoints(
    crop_for_viz, kp, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)
crop_with_kp = cv2.cvtColor(crop_with_kp, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(crop_with_kp)
ax.set_title(f"SIFT: {len(kp)} keypoints en {first_label}")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. BoVW

Asignamos cada SIFT al centroide más cercano del codebook (K=1500) y contamos. El histograma resultante es la "firma" visual del crop, de longitud fija K independientemente de cuántos keypoints tenga el objeto.

In [ ]:
if not codebook_path.exists():
    raise FileNotFoundError(
        f"Codebook no encontrado en {codebook_path}.\n"
        "Ejecuta:  uv run python scripts/train_codebook.py"
    )

codebook = load_codebook(codebook_path)
print(f"Codebook K={codebook.n_clusters}")

bovw_hist = encode_bovw(desc, codebook)
print(f"BoVW histogram: shape={bovw_hist.shape}  L2 norm={np.linalg.norm(bovw_hist):.4f}")
print(f"Active bins (count>0): {(bovw_hist > 0).sum()} / {bovw_hist.size}")

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(np.arange(bovw_hist.size), bovw_hist, width=1.0, color="steelblue")
ax.set_xlim(0, bovw_hist.size)
ax.set_xlabel("visual word id")
ax.set_ylabel("L2-norm count")
ax.set_title(f"BoVW histogram — {first_label}")
plt.tight_layout()
plt.show()

## 5. Feature final concatenado

El vector que recibe la SVM en H4 es la concatenación de HOG (forma global) y BoVW (textura/palabras visuales), normalizado L2.

In [ ]:
feature = np.concatenate([hog_vec, bovw_hist]).astype(np.float32)
norm = np.linalg.norm(feature)
if norm > 0:
    feature = feature / norm
print(f"Feature final: shape={feature.shape}")
print(f"  HOG part : {hog_vec.shape[0]} dims  (índices 0..{hog_vec.shape[0]-1})")
print(f"  BoVW part: {bovw_hist.shape[0]} dims  (índices {hog_vec.shape[0]}..{feature.shape[0]-1})")
print(f"  L2 norm  : {np.linalg.norm(feature):.4f}")

## 6. Tiempo end-to-end por proposal

Para dimensionar H4: ¿cuánto tarda extraer la feature completa para un proposal? Si SS produce ~1000 proposals/imagen y tenemos 1650 train + 1407 test, el coste total importa.

In [ ]:
n_test = 50
sample_props = props[:n_test]

t0 = time.time()
for (x, y, w, h) in sample_props:
    bb = (int(x), int(y), int(w), int(h))
    h_vec = compute_hog(img_bgr, bbox=bb)
    _, d = compute_sift(img_bgr, bbox=bb)
    b_vec = encode_bovw(d, codebook)
    f = np.concatenate([h_vec, b_vec])
elapsed = time.time() - t0
per_prop_ms = 1000 * elapsed / n_test
print(f"{n_test} proposals procesadas en {elapsed:.2f}s  ->  {per_prop_ms:.1f} ms/proposal")
print(f"Proyección 1000 proposals/imagen: {per_prop_ms:.0f} ms x 1000 = {per_prop_ms:.1f}s/imagen")
print(f"Train (1650 imgs): ~{per_prop_ms * 1650 / 1000 / 60:.1f} min de extracción de features")
print(f"Test  (1407 imgs): ~{per_prop_ms * 1407 / 1000 / 60:.1f} min de extracción de features")

## 7. Resumen H3

**Cerrado:**

- `src/grocery_detection/classical/proposals.py` — Selective Search fast/quality con resize + rescale.
- `src/grocery_detection/classical/descriptors/hog.py` — HOG sobre crop 64x64.
- `src/grocery_detection/classical/descriptors/sift.py` — SIFT keypoints + descriptores 128-D.
- `src/grocery_detection/classical/descriptors/bovw.py` — codebook k-means + encoding L2.
- `scripts/train_codebook.py` — CLI que entrena el codebook desde train sampling.
- `data/processed/codebook.pkl` — codebook entrenado.

**Próximo: H4** — entrenar las 20 SVMs one-vs-rest con kernel chi² sobre proposals positivos (IoU>=0.5 con GT) + negativos. Inferencia sobre test con filtrado por threshold + NMS. Output: predicciones COCO JSON sobre test.